# Silver region population

## Purpose

Create the clean region-level population input for the future Gold population fact. One output row represents one official PSGC region for the July 1, 2024 Table C reference, source, and selected snapshots. The natural key is region code, reference year, source name, Table C snapshot, and PSGC snapshot. Silver selects the non-overlapping population grain and preserves lineage; it does not create Gold keys or analytical ratios.


## 1. Enforce the reconciliation gate

Every current Table C row must appear exactly once in reconciliation before aggregation. Accepted primary rows must be positive matched barangays. These stop checks prevent incomplete or overlapping population records from reaching the regional table.


In [0]:
USE CATALOG `buildabida-capstone`;

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `02-silver`.silver_population_place_reconciliation)
        = (SELECT COUNT(*) FROM `01-bronze`.census_2024_table_c),
    'Population reconciliation must retain exactly one row for every current Table C row'
);

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'source_file', source_file,
        'sheet_name', sheet_name,
        'source_row_number', source_row_number,
        'run_id', run_id
    )),
    'Population reconciliation contains duplicate source-row grain'
)
FROM `02-silver`.silver_population_place_reconciliation;

SELECT ASSERT_TRUE(
    COUNT_IF(
        row_disposition = 'ACCEPTED_PRIMARY'
        AND (population_count <= 0 OR matched_place_type <> 'BARANGAY' OR matched_region_code IS NULL)
    ) = 0,
    'Accepted population rows must be positive matched barangays with an official region'
)
FROM `02-silver`.silver_population_place_reconciliation;

SELECT ASSERT_TRUE(
    COUNT_IF(row_disposition = 'ACCEPTED_PRIMARY') > 0,
    'At least one positive matched barangay is required for region population'
)
FROM `02-silver`.silver_population_place_reconciliation;


## 2. Aggregate only the approved non-overlapping grain

The authoritative population input is Table C. Only `ACCEPTED_PRIMARY` barangay rows contribute, which avoids adding province and locality subtotals to barangay population. Region membership comes from the matched PSGC hierarchy. PSGC population remains diagnostic and is not aggregated as the authoritative measure. The official selected PSGC region set drives coverage, so no fixed region count is hardcoded. Missing region population remains null rather than becoming zero.


In [0]:
CREATE OR REPLACE TABLE `02-silver`.silver_region_population
USING DELTA
COMMENT 'Authoritative Table C population aggregated from accepted matched barangays to official PSGC regions'
AS
WITH lineage AS (
    SELECT
        MAX(reference_year) AS reference_year,
        MAX(population_reference_date) AS population_reference_date,
        MAX(source_name) AS source_name,
        MAX(source_snapshot_id) AS source_snapshot_id,
        MAX(source_ingest_run_id) AS source_ingest_run_id,
        MAX(psgc_source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(psgc_source_ingest_run_id) AS psgc_source_ingest_run_id,
        MAX(psgc_version) AS psgc_version,
        MAX(alias_version) AS alias_version,
        MAX(run_id) AS population_reconciliation_run_id
    FROM `02-silver`.silver_population_place_reconciliation
),
official_regions AS (
    SELECT
        place.psgc_code AS psgc_region_code,
        place.place_name_raw AS region_name,
        place.source_snapshot_id AS psgc_source_snapshot_id
    FROM `02-silver`.silver_psgc_place AS place
    CROSS JOIN lineage
    WHERE place.place_type = 'REGION'
      AND place.source_snapshot_id = lineage.psgc_source_snapshot_id
),
population_by_region AS (
    SELECT
        matched_region_code AS psgc_region_code,
        SUM(population_count) AS population_count,
        COUNT(*) AS contributing_source_row_count
    FROM `02-silver`.silver_population_place_reconciliation
    WHERE row_disposition = 'ACCEPTED_PRIMARY'
      AND matched_place_type = 'BARANGAY'
      AND population_count > 0
    GROUP BY matched_region_code
)
SELECT
    region.psgc_region_code,
    region.region_name,
    lineage.reference_year,
    lineage.population_reference_date,
    population.population_count,
    lineage.source_name,
    TRUE AS is_primary_source,
    CASE WHEN population.population_count IS NULL THEN 'MISSING_PRIMARY_POPULATION' ELSE 'MATCHED_PRIMARY_POPULATION' END
        AS population_match_status,
    COALESCE(population.contributing_source_row_count, 0) AS source_row_count,
    lineage.source_snapshot_id,
    lineage.source_ingest_run_id,
    lineage.psgc_source_snapshot_id,
    lineage.psgc_source_ingest_run_id,
    lineage.psgc_version,
    lineage.alias_version,
    'silver-region-population-v1' AS transformation_rule_version,
    SHA2(CONCAT_WS(
        '|',
        lineage.population_reconciliation_run_id,
        'silver-region-population-v1'
    ), 256) AS run_id,
    lineage.population_reconciliation_run_id
FROM official_regions AS region
CROSS JOIN lineage
LEFT JOIN population_by_region AS population
    ON region.psgc_region_code = population.psgc_region_code;


## 3. Reconcile the regional result

The accepted Table C barangay total must equal the regional output total. The query also shows official region coverage and retains missing populations as null. It does not compare against a total that includes higher-level subtotals.


In [0]:
WITH accepted AS (
    SELECT
        COUNT(*) AS accepted_source_rows,
        SUM(population_count) AS accepted_population
    FROM `02-silver`.silver_population_place_reconciliation
    WHERE row_disposition = 'ACCEPTED_PRIMARY'
),
regional AS (
    SELECT
        COUNT(*) AS official_region_rows,
        COUNT_IF(population_count IS NULL) AS missing_population_regions,
        SUM(source_row_count) AS contributing_source_rows,
        SUM(population_count) AS regional_population
    FROM `02-silver`.silver_region_population
)
SELECT
    accepted.accepted_source_rows,
    regional.contributing_source_rows,
    regional.official_region_rows,
    regional.missing_population_regions,
    accepted.accepted_population,
    regional.regional_population,
    regional.regional_population - accepted.accepted_population AS reconciliation_difference
FROM accepted
CROSS JOIN regional;

SELECT ASSERT_TRUE(
    (
        SELECT SUM(population_count)
        FROM `02-silver`.silver_population_place_reconciliation
        WHERE row_disposition = 'ACCEPTED_PRIMARY'
    ) = (SELECT SUM(population_count) FROM `02-silver`.silver_region_population),
    'Regional population must reconcile exactly to accepted matched Table C barangay rows'
);


# Summary

Created `02-silver.silver_region_population` at one official PSGC region per reference year, source, and selected snapshots. It aggregates only positive accepted matched Table C barangays and derives region membership from PSGC. The table preserves both upstream snapshots, both Bronze ingest runs, alias and rule versions, reconciliation run identity, and its own deterministic Silver run. Missing official-region population stays null. Requires Databricks execution. Next dependency: Silver geography and population validation, followed later by Gold dimension and fact work.
